# **Notebook for data pre-processing and cleanning**


### Imports

In [1]:
import geopandas as gpd
import osmnx as ox
import numpy as np
import networkx as nx
import pandas as pd
from shapely.ops import unary_union
import contextily as cx
import matplotlib.pyplot as plt
from tqdm import tqdm
import h3

%matplotlib inline
ox.__version__

'1.9.4'

## Data Loading

### Addresses data

This data was extracted from: dataforsyningen.dk

The documentation is availible at: https://dawadocs.dataforsyningen.dk/dok/api/adresse#databeskrivelse

To download type in your browser: https://api.dataforsyningen.dk/adresser?kommunekode=0101|0147&format=csv


In [ ]:
PATH_0101 = "../data/adresser.csv"
PATH_01010147 = "../data/adresser_0101_0147.csv"

df_address_copen = gpd.read_file(PATH_0101)
print(f"Shape of df_address_copen: {df_address_copen.shape}")
df_address = gpd.read_file(PATH_01010147)
print(f"Shape of df_address: {df_address.shape}")
df_address.head()

(510681, 21)


,id,status,darstatus,vejkode,vejnavn,adresseringsvejnavn,husnr,etage,dÃ¸r,supplerendebynavn,...,postnrnavn,stormodtagerpostnr,stormodtagerpostnrnavn,kommunekode,adgangsadresseid,x,y,href,betegnelse,geometry
0,76a258f3-8deb-4174-a62b-c845153958f0,1,3,7932,ValgÃ¥rdsvej,ValgÃ¥rdsvej,2,,,,...,Valby,,,0101,0a3f507b-1cc7-32b8-e044-0003ba298018,12.51855172,55.66143536,https://api.dataforsyningen.dk/adresser/76a258...,"ValgÃ¥rdsvej 2, 2500 Valby",None
1,0a3f509d-24c0-32b8-e044-0003ba298018,1,3,0116,Amagergade,Amagergade,4C,,,,...,KÃ¸benhavn K,,,0101,0a3f507a-3a78-32b8-e044-0003ba298018,12.5929662,55.67062106,https://api.dataforsyningen.dk/adresser/0a3f50...,"Amagergade 4C, 1423 KÃ¸benhavn K",None
2,a68c1135-4fe6-4263-89f8-08431fd0bb9c,1,3,7334,TaffelÃ¦blevej,TaffelÃ¦blevej,6,1,th,,...,Valby,,,0101,000709e1-ff74-4199-9798-f0c39b835da5,12.5035067,55.65615644,https://api.dataforsyningen.dk/adresser/a68c11...,"TaffelÃ¦blevej 6, 1. th, 2500 Valby",None
3,e2b57dba-c49e-48bc-8783-dcd1082680db,1,3,8841,Dieselvej,Dieselvej,14,1,tv,,...,KÃ¸benhavn SV,,,0101,0016c4dd-5bee-4b2e-b5a4-72fc6ae82519,12.55527079,55.6513282,https://api.dataforsyningen.dk/adresser/e2b57d...,"Dieselvej 14, 1. tv, 2450 KÃ¸benhavn SV",None
4,ec376d97-6cae-4a0f-833c-706b9b5a48df,1,3,0497,Lauritz-Jensens Plads,Lauritz-Jensens Pl.,1,,,,...,Frederiksberg,,,0147,bc6eaf28-9b51-451f-b282-bb0c3c52f534,12.49887067,55.68160979,https://api.dataforsyningen.dk/adresser/ec376d...,"Lauritz-Jensens Plads 1, 2000 Frederiksberg",None


In [4]:
df_address["geometry"] = gpd.points_from_xy(df_address.x, df_address.y)
address_gdf = gpd.GeoDataFrame(df_address, geometry="geometry", crs="EPSG:4258")
print(address_gdf['kommunekode'].unique())
print(address_gdf.shape)
address_gdf.head(3)

['0101' '0147']
(510681, 21)


,id,status,darstatus,vejkode,vejnavn,adresseringsvejnavn,husnr,etage,dÃ¸r,supplerendebynavn,...,postnrnavn,stormodtagerpostnr,stormodtagerpostnrnavn,kommunekode,adgangsadresseid,x,y,href,betegnelse,geometry
0,76a258f3-8deb-4174-a62b-c845153958f0,1,3,7932,ValgÃ¥rdsvej,ValgÃ¥rdsvej,2,,,,...,Valby,,,0101,0a3f507b-1cc7-32b8-e044-0003ba298018,12.51855172,55.66143536,https://api.dataforsyningen.dk/adresser/76a258...,"ValgÃ¥rdsvej 2, 2500 Valby",POINT (12.51855 55.66144)
1,0a3f509d-24c0-32b8-e044-0003ba298018,1,3,0116,Amagergade,Amagergade,4C,,,,...,KÃ¸benhavn K,,,0101,0a3f507a-3a78-32b8-e044-0003ba298018,12.5929662,55.67062106,https://api.dataforsyningen.dk/adresser/0a3f50...,"Amagergade 4C, 1423 KÃ¸benhavn K",POINT (12.59297 55.67062)
2,a68c1135-4fe6-4263-89f8-08431fd0bb9c,1,3,7334,TaffelÃ¦blevej,TaffelÃ¦blevej,6,1,th,,...,Valby,,,0101,000709e1-ff74-4199-9798-f0c39b835da5,12.5035067,55.65615644,https://api.dataforsyningen.dk/adresser/a68c11...,"TaffelÃ¦blevej 6, 1. th, 2500 Valby",POINT (12.50351 55.65616)


In [ ]:
## Plot over a map
address_gdf.explore()

> [!NOTE]
> @Caroline look into cleanning this to leave only the valid addresses and houses